<a href="https://colab.research.google.com/github/internshipinabook/nlp-internship-in-a-book/blob/main/notebooks/week9_fine_tuning_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⏸️ **Pause and Predict**
> Before fine-tuning: what do you expect to happen to training loss and validation loss across epochs? Draw a rough sketch of your prediction. Compare to the actual learning curves.

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/nlp-internship-in-a-book.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 9 — Fine-Tuning for the Real World (STARTER)
### The NLP Internship | LinguaAI Labs

**The genuine cross-lingual transfer experiment:**
- Train AfroXLMR on English CLINC150
- Evaluate zero-shot on Yorùbá MASSIVE (yo-NG)
- ΔF1 = F1(English) − F1(Yorùbá) = the cross-lingual transfer gap

> ⏸️ **Predict** ΔF1 before running: < 0.10 (excellent) | 0.10–0.25 (expected) | > 0.25 (poor)?

In [ ]:
import sys, subprocess, os, torch
for pkg in ["datasets","transformers"]:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           Trainer, TrainingArguments)
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, classification_report
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42); os.makedirs("outputs",exist_ok=True); os.makedirs("models",exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

DOMAIN_MAP = {
    "bill_balance":"billing","bill_pay":"billing","transfer":"billing","pay_bill":"billing",
    "account_blocked":"billing","min_payment":"billing",
    "freeze_account":"account","lost_stolen_card":"account","report_fraud":"account",
    "card_declined":"account","cancel_card":"account","credit_score":"account",
    "travel_alert":"travel","flight_status":"travel","book_hotel":"travel","book_flight":"travel",
    "reminder":"utility","calendar":"utility","schedule_meeting":"utility","make_call":"utility",
}
print("Loading CLINC150 (en-US) — training corpus...")
clinc = load_dataset("clinc_oos", "plus")
label_names_clinc = clinc["train"].features["intent"].names
def collapse_clinc(i): return DOMAIN_MAP.get(label_names_clinc[i], "other")
df_train = clinc["train"].to_pandas().rename(columns={"text":"text_clean"})
df_test  = clinc["test"].to_pandas().rename(columns={"text":"text_clean"})
df_train["category"] = df_train["intent"].apply(collapse_clinc)
df_test["category"]  = df_test["intent"].apply(collapse_clinc)
X_train=df_train["text_clean"].values; y_train_str=df_train["category"].values
X_test =df_test["text_clean"].values;  y_test_str =df_test["category"].values
label2id={l:i for i,l in enumerate(sorted(set(y_train_str)))}
id2label={v:k for k,v in label2id.items()}
y_train=np.array([label2id[l] for l in y_train_str])
y_test =np.array([label2id[l] for l in y_test_str])

print("Loading MASSIVE yo-NG — Yorùbá cross-lingual eval corpus...")
massive = load_dataset("AmazonScience/massive", "yo-NG")
MASSIVE_DOMAIN_MAP = {
    "alarm_set":"utility","alarm_query":"utility","calendar_set":"utility",
    "calendar_query":"utility","reminder_set":"utility","reminder_query":"utility",
    "transport_query":"travel","transport_taxi":"travel","transport_ticket":"travel",
    "qa_factoid":"other","general_quirky":"other","play_music":"other",
}
label_names_massive = massive["test"].features["intent"].names
def collapse_massive(i): return MASSIVE_DOMAIN_MAP.get(label_names_massive[i], "other")
df_yo = massive["test"].to_pandas()
df_yo["category"] = df_yo["intent"].apply(collapse_massive)
df_yo = df_yo.rename(columns={"utt":"text_clean"})
known = set(label2id.keys())
df_yo_known = df_yo[df_yo["category"].isin(known)].copy()
X_yo = df_yo_known["text_clean"].values
y_yo = df_yo_known["category"].values

print(f"CLINC150 train: {len(X_train):,} | MASSIVE yo-NG: {len(X_yo):,} Yorùbá utterances ✅")

## Part 1 — Tokenisation Comparison

In [ ]:
bert_tok  = AutoTokenizer.from_pretrained("bert-base-uncased")
mbert_tok = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
afro_tok  = AutoTokenizer.from_pretrained("Davlan/afro-xlmr-mini")

test_sents = [("CLINC en",X_test[0]),("MASSIVE yo",X_yo[0]),("MASSIVE yo",X_yo[1])]
print(f"{'Text type':<14} {'BERT-en':>9} {'mBERT':>8} {'AfroXLMR':>10}")
print("-"*45)
for label, sent in test_sents:
    print(f"  {label:<12} {len(bert_tok.tokenize(sent)):>9}"
          f" {len(mbert_tok.tokenize(sent)):>8}"
          f" {len(afro_tok.tokenize(sent)):>10}")
print("\nFewer AfroXLMR tokens = better Yorùbá vocabulary coverage.")

## Part 2 — Fine-Tune AfroXLMR on CLINC150

In [ ]:
afro_model_name = "Davlan/afro-xlmr-mini"
afro_tok_model  = AutoTokenizer.from_pretrained(afro_model_name)
def tok_fn(ex): return afro_tok_model(ex["text"],padding="max_length",truncation=True,max_length=128)
train_ds = Dataset.from_dict({"text":list(X_train),"label":list(y_train)}).map(tok_fn,batched=True)
test_ds  = Dataset.from_dict({"text":list(X_test), "label":list(y_test)}).map(tok_fn,batched=True)

# YOUR CODE HERE — AutoModelForSequenceClassification.from_pretrained(afro_model_name, ...)
afro_model = None  # YOUR CODE HERE

def compute_metrics(ep): return {"f1":f1_score(ep.label_ids,ep.predictions.argmax(-1),average="weighted",zero_division=0)}
args = TrainingArguments(output_dir="models/afro_xlmr_clinc",num_train_epochs=3,
    per_device_train_batch_size=32,per_device_eval_batch_size=64,
    warmup_steps=100,weight_decay=0.01,evaluation_strategy="epoch",
    save_strategy="epoch",load_best_model_at_end=True,metric_for_best_model="f1",seed=42)
# YOUR CODE HERE — Trainer(...) and trainer.train()

## Part 3 — English Evaluation

In [ ]:
# YOUR CODE HERE — trainer.predict(test_ds)
preds_en = None  # YOUR CODE HERE
y_pred_en = None  # YOUR CODE HERE
en_f1 = None  # YOUR CODE HERE
print(f"English (CLINC): F1 = {en_f1:.3f}")
print(classification_report([id2label[i] for i in y_test],[id2label[i] for i in y_pred_en],zero_division=0))

## Part 4 — Zero-Shot Yorùbá Evaluation (MASSIVE yo-NG)

> 🧠 **Predict Yorùbá F1 before running.** Model trained on English only.

In [ ]:
yo_ds = Dataset.from_dict({"text":list(X_yo),
                            "label":[label2id[y] for y in y_yo]}).map(tok_fn,batched=True)
# YOUR CODE HERE — trainer.predict(yo_ds)
preds_yo = None  # YOUR CODE HERE
y_pred_yo = None  # YOUR CODE HERE
yo_f1 = None  # YOUR CODE HERE

print(f"Cross-lingual transfer results:")
print(f"  English (CLINC en-US):  F1 = {en_f1:.3f}  [trained on this]")
print(f"  Yorùbá  (MASSIVE yo-NG): F1 = {yo_f1:.3f}  [zero-shot]")
print(f"  Transfer gap ΔF1:       {en_f1 - yo_f1:.3f}")
delta = en_f1 - yo_f1 if (en_f1 and yo_f1) else 0
if   delta < 0.10: print("  → Excellent transfer")
elif delta < 0.25: print("  → Expected gap")
else:              print("  → Poor transfer — consider Yorùbá-specific data")
print("\n📚 FitzGerald et al. (2022) MASSIVE arXiv:2204.08582 — Apache 2.0")
print("📚 Muhammad et al. (2023) AfriSenti EMNLP — CC BY 4.0")

## 🏆 Challenge Task

> Compare mBERT vs AfroXLMR: fine-tune both on CLINC150 with identical hyperparameters, evaluate both on MASSIVE yo-NG. Which benefits more from African language pre-training on an unseen Yorùbá task?

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Training for too many epochs | After 3-5 epochs, most transformer models on small datasets begin to overfit. Longer training does not always mean better models. | Monitor validation loss every epoch. Stop when it stops decreasing. |
| Evaluating only on the final epoch | The best model is not always the last model. Save checkpoints and evaluate each one. | Use save_best_only=True or equivalent — never assume the last checkpoint is the best |

## ✅ What You Can Do After This Week

- Run a genuine cross-lingual transfer experiment on published datasets
- Fine-tune AfroXLMR on English and evaluate zero-shot on Yorùbá
- Interpret ΔF1 as a research metric
- Cite CLINC150, MASSIVE, and AfriSenti correctly in a paper

---
## ✅ Week 9 Complete — Next: `week10_fairness_STARTER.ipynb`

---
*Add `week9_fine_tuning_STARTER.ipynb` to your portfolio.*

*The NLP Internship · LinguaAI Labs · github.com/InternshipInABook*

## ✅ By completing Week 9, you can now:

- Fine-tune a transformer model and monitor training vs validation loss
- Implement early stopping and explain why it matters
- Evaluate a fine-tuned model against a zero-shot baseline rigorously
- Save a complete fine-tuned pipeline ready for deployment

📤 **GitHub:** Push week9_fine_tuning_STARTER.ipynb. Commit: "Week 9: AfroXLMR fine-tuned, early stopping implemented"


---

## 📚 Get the Full Book

This notebook is part of **The NLP Internship** — Book 2 of the InternshipInABook™ Series.

The full book includes:
- Complete week-by-week narrative and explanations
- All STOP AND TRACE code walkthroughs
- Fairness briefs, model cards, and deployment guides
- Certificate of Completion

👉 **[Get the book on Selar →](https://selar.com/8440iqfr61)**

---
*InternshipInABook™ Series · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook)*
